In [0]:
%pip install yfinance

import yfinance as yf
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

In [0]:
# reference spark session
spark = SparkSession.builder.getOrCreate()

In [0]:
# extract stock data for Apple (AAPL) from Yahoo Finance
ticker = "AAPL"
df = yf.download(ticker, period="2y", interval="1d")
df = df.reset_index()

In [0]:
df.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in df.columns]

In [0]:
print(df.head())

In [0]:
df["ticker"] = ticker
df["ingestion_time"] = pd.Timestamp.now()

In [0]:
df = df.rename(columns={
    "Date_": "date",
    "Open_AAPL": "open",
    "High_AAPL": "high",
    "Low_AAPL": "low",
    "Close_AAPL": "close",
    "Volume_AAPL": "volume"
})

In [0]:
# convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
# write to bronze data table
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.aapl_stock_data")

print(f'Successfully ingested {ticker} stock data to bronze table.')